# Returns and Corporate-Action Forensics
## Project synthesis

### Research question
How severely can incorrect corporate-action treatment distort measured
returns, risk diagnostics, technical signals, cumulative wealth, and
statistical models?

### Thesis

A market-data field is meaningful only when its economic unit is understood.
Stock splits change the number of shares and the price quoted per share,
while cash dividends transfer value from the company to the shareholder.

If these events are treated as ordinary price movements:

- splits can appear as losses
- dividends can appear as negative investor returns
- rolling statistics remain contaminated after the event
- technical indicators can compare incompatible price units
- supervised-learning features and targets can acquire false economic labels

The project combines deterministic experiments with real Apple and Coca-Cola
corporate actions.

# 1. Evidence map

## Experiment A — Synthetic split and model contamination

A deterministic 2-for-1 split isolates the mechanical accounting effect.
The experiment compares raw and action-aware returns, rolling volatility,
z-scores, moving-average signals, and ridge-regression diagnostics.

Primary notebook:

`01_synthetic_split.ipynb`

## Experiment B — Cash-dividend accounting

A synthetic dividend example establishes the total-return identity. Real
Coca-Cola data compare price return, explicit dividend accounting, and
vendor-adjusted close. Alpha Vantage independently verifies the yfinance
event dates and amounts.

Primary notebook:

`02_cash_dividend.ipynb`

## Experiment C — Real Apple stock split

Apple's 2020 4-for-1 split is used to reconstruct historical quote units,
compare naive and split-aware returns, and reconcile price-per-share changes
with shareholder wealth.

Primary notebook:

`03_real_corporate_actions.ipynb`

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root containing pyproject.toml"
    )


project_root = find_project_root(Path.cwd())
reports_dir = project_root / "reports"
tables_dir = reports_dir / "tables"
figures_dir = reports_dir / "figures"

# 2. Load existing evidence

The synthesis notebook should not duplicate the calculations performed in
the experiment notebooks. Instead, it reads their saved report artifacts.

This separation has two advantages:

1. each experiment remains the authoritative source for its calculations
2. the synthesis can be regenerated from compact, auditable outputs

In [2]:
synthetic_model_path = (
    tables_dir
    / "synthetic_split_model_distortion.csv"
)

dividend_vendor_path = (
    tables_dir
    / "ko_dividend_vendor_comparison.csv"
)

apple_split_path = (
    tables_dir
    / "aapl_real_split_evidence.csv"
)
synthetic_summary_path = (
    tables_dir
    / "synthetic_split_evidence.csv"
)

dividend_summary_path = (
    tables_dir
    / "ko_dividend_evidence.csv"
)
required_paths = [
    synthetic_model_path,
    dividend_vendor_path,
    apple_split_path,
    synthetic_summary_path,
    dividend_summary_path
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        f"Missing required report artifacts:\n{missing_text}"
    )

In [3]:
synthetic_model_evidence = pd.read_csv(
    synthetic_model_path,
    index_col=0,
)

dividend_vendor_evidence = pd.read_csv(
    dividend_vendor_path,
)

apple_split_evidence = pd.read_csv(
    apple_split_path,
    index_col=0,
)

synthetic_summary = (
    pd.read_csv(
        synthetic_summary_path,
        index_col=0,
    )["value"]
)

dividend_summary = (
    pd.read_csv(
        dividend_summary_path,
        index_col=0,
    )["value"]
)

apple_summary = apple_split_evidence["value"]

In [4]:
artifact_manifest = pd.DataFrame(
    {
        "artifact": [
            "Synthetic split evidence",
            "Synthetic model distortion",
            "KO dividend evidence",
            "KO dividend vendor comparison",
            "Apple real split evidence",
        ],
        "rows": [
            len(synthetic_summary),
            len(synthetic_model_evidence),
            len(dividend_summary),
            len(dividend_vendor_evidence),
            len(apple_summary),
        ],
        "source_file": [
            synthetic_summary_path.name,
            synthetic_model_path.name,
            dividend_summary_path.name,
            dividend_vendor_path.name,
            apple_split_path.name,
        ],
    }
)

display(artifact_manifest)

,artifact,rows,source_file
0,Synthetic split evidence,11,synthetic_split_evidence.csv
1,Synthetic model distortion,4,synthetic_split_model_distortion.csv
2,KO dividend evidence,9,ko_dividend_evidence.csv
3,KO dividend vendor comparison,12,ko_dividend_vendor_comparison.csv
4,Apple real split evidence,8,aapl_real_split_evidence.csv


# 3. Cross-experiment distortion table

The experiments use different units, so the table does not combine them into
a single score. Instead, it reports the principal incorrect measurement,
the corresponding action-aware result, and the size of the distortion.

The purpose is to show that corporate-action errors propagate through several
layers of a research pipeline: returns, rolling diagnostics, trading signals,
cumulative wealth, and statistical models.

In [5]:
def format_percent(value: object) -> str:
    return f"{100 * float(value):.2f}%"


cross_experiment_distortion = pd.DataFrame(
    [
        {
            "experiment": "Synthetic 2-for-1 split",
            "research_object": "Split-date return",
            "incorrect_treatment": format_percent(
                synthetic_summary["naive_split_return"]
            ),
            "action_aware_result": format_percent(
                synthetic_summary["split_aware_return"]
            ),
            "distortion": "False 50 percentage-point loss",
        },
        {
            "experiment": "Synthetic 2-for-1 split",
            "research_object": "20-day rolling volatility",
            "incorrect_treatment": (
                f"{float(synthetic_summary['peak_volatility_distortion']):.2f}"
                "× action-aware volatility"
            ),
            "action_aware_result": "Economic-return volatility",
            "distortion": "One event contaminates a full rolling window",
        },
        {
            "experiment": "Synthetic 2-for-1 split",
            "research_object": "Ex-ante return z-score",
            "incorrect_treatment": (
                f"{float(synthetic_summary['raw_split_zscore']):.2f}"
            ),
            "action_aware_result": (
                f"{float(synthetic_summary['split_aware_zscore']):.2f}"
            ),
            "distortion": "Bookkeeping event appears as an extreme anomaly",
        },
        {
            "experiment": "Synthetic model contamination",
            "research_object": "Ridge-regression RMSE",
            "incorrect_treatment": (
                f"{float(synthetic_summary['raw_model_rmse']):.4f}"
            ),
            "action_aware_result": (
                f"{float(synthetic_summary['split_aware_model_rmse']):.4f}"
            ),
            "distortion": (
                f"{float(synthetic_summary['raw_model_rmse']) / float(synthetic_summary['split_aware_model_rmse']):.2f}"
                "× higher RMSE"
            ),
        },
        {
            "experiment": "Coca-Cola dividends",
            "research_object": "Dividend-date return sign",
            "incorrect_treatment": (
                f"{int(float(dividend_summary['sign_reversal_dates']))} "
                "apparent price-only losses"
            ),
            "action_aware_result": "Non-negative total returns",
            "distortion": (
                f"{100 * float(dividend_summary['sign_reversal_share']):.1f}% "
                "of dividend dates"
            ),
        },
        {
            "experiment": "Coca-Cola dividends",
            "research_object": "Three-year wealth",
            "incorrect_treatment": (
                f"{float(dividend_summary['price_only_growth']):.3f}"
            ),
            "action_aware_result": (
                f"{float(dividend_summary['total_return_growth']):.3f}"
            ),
            "distortion": (
                f"{100 * float(dividend_summary['total_wealth_premium_vs_price_only']):.2f}% "
                "higher total-return wealth"
            ),
        },
        {
            "experiment": "Apple 4-for-1 split",
            "research_object": "Split-date return",
            "incorrect_treatment": format_percent(
                apple_summary["naive_price_return"]
            ),
            "action_aware_result": format_percent(
                apple_summary["split_aware_return"]
            ),
            "distortion": "Positive day appears as a 74% collapse",
        },
        {
            "experiment": "Dividend vendor verification",
            "research_object": "Event dates and amounts",
            "incorrect_treatment": "Single-vendor event records before verification",
            "action_aware_result": (
                f"{int(float(dividend_summary['matched_vendor_event_dates']))} "
                "of 12 events matched"
            ),
            "distortion": (
                f"Maximum amount difference: "
                f"{float(dividend_summary['maximum_vendor_amount_difference']):.3f}"
            ),
        },
    ]
)

display(cross_experiment_distortion)

,experiment,research_object,incorrect_treatment,action_aware_result,distortion
0,Synthetic 2-for-1 split,Split-date return,-50.00%,0.00%,False 50 percentage-point loss
1,Synthetic 2-for-1 split,20-day rolling volatility,13.63× action-aware volatility,Economic-return volatility,One event contaminates a full rolling window
2,Synthetic 2-for-1 split,Ex-ante return z-score,-46.17,0.19,Bookkeeping event appears as an extreme anomaly
3,Synthetic model contamination,Ridge-regression RMSE,0.0656,0.0102,6.44× higher RMSE
4,Coca-Cola dividends,Dividend-date return sign,4 apparent price-only losses,Non-negative total returns,33.3% of dividend dates
5,Coca-Cola dividends,Three-year wealth,1.111,1.215,9.42% higher total-return wealth
6,Apple 4-for-1 split,Split-date return,-74.15%,3.39%,Positive day appears as a 74% collapse
7,Dividend vendor verification,Event dates and amounts,Single-vendor event records before verification,12 of 12 events matched,Maximum amount difference: 0.000


In [6]:
distortion_table_path = (
    tables_dir
    / "project_cross_experiment_distortion.csv"
)

cross_experiment_distortion.to_csv(
    distortion_table_path,
    index=False,
)

### Interpretation

The largest distortions occur when a corporate action changes the economic
unit represented by the price series.

In the synthetic split, one mechanical event creates a false $-50\%$
return, inflates rolling volatility by more than thirteen times, generates
an ex-ante z-score near $-46$, and increases ridge-regression RMSE by more
than six times. The error persists beyond the event because rolling
estimators and lagged model features continue to contain the contaminated
observation.

The real Apple split shows the same mechanism in market data. A naive
comparison of historical quote units reports a $-74.15\%$ return, while
share-count accounting and the vendor's split-adjusted close both show an
economic return of $+3.39\%$.

Dividend errors are smaller on individual dates but accumulate through time.
For Coca-Cola, ignoring dividends leaves three-year wealth approximately
$9.42\%$ below dividend-inclusive wealth. On four of twelve dividend dates,
price-only return reports a loss while total return is non-negative.

The independent vendor comparison matched all twelve ex-dividend dates and
amounts, strengthening confidence that the dividend conclusions are not
caused by missing or shifted corporate-action records.

## Main conclusions

### 1. Prices must be interpreted together with their share unit

A raw price change is not automatically an investor return. Splits alter the
number of shares and the quoted price per share, so prices across the event
cannot be compared without adjusting either the price series or the share
count.

### 2. Total return is the relevant wealth measure

Cash dividends transfer value to shareholders. A price-only return omits that
cash flow and can misstate both the magnitude and the sign of the investor's
return.

### 3. Corporate-action errors propagate downstream

One incorrect return can alter rolling volatility, anomaly scores, technical
signals, feature scaling, model targets, residual distributions, and reported
model performance. Correcting the data after model fitting is too late.

### 4. Robust statistics do not repair incorrect economic labels

Robust scaling reduced the influence of the synthetic split on the feature
center and scale, but it did not repair the false regression target. Economic
validation must precede statistical robustness.

### 5. Adjusted prices are analytical constructs

Adjusted prices create return continuity and are useful for research, but
they are not necessarily the prices historically displayed or available for
execution. Raw prices, action fields, and adjustment conventions should be
preserved separately.

# 4. Limitations

## Data limitations

- The real-data experiments use yfinance and Alpha Vantage, which are
  appropriate for educational research but are not institutional-grade
  historical databases.
- Exact agreement between vendors strengthens confidence but does not prove
  complete independence because vendors may rely on overlapping upstream
  data sources.
- Historical records may be revised after the original event date.
- The reconstructed Apple as-traded series is derived from split-adjusted
  prices and the reported split ratio rather than from an archival raw-price
  database.

## Economic-convention limitations

- Total-return calculations assume dividends remain invested and ignore
  taxes, withholding, commissions, and reinvestment frictions.
- A split has no mechanical effect on wealth, but the stock can still move
  for ordinary market reasons during the same return interval.
- Adjusted-close conventions can differ across vendors, particularly in
  treatment of dividends, special distributions, and rounding.

## Statistical limitations

- The synthetic experiments use one deterministic simulation and are designed
  to demonstrate mechanisms rather than estimate population effects.
- The ridge-regression experiment is an in-sample contamination diagnostic,
  not evidence of return predictability.
- Rolling-window distortion depends on the chosen window length.
- Results from Apple and Coca-Cola should not be generalized to every
  security or every type of corporate action.

## Scope limitations

This project studies ordinary cash dividends and stock splits. It does not
fully address spin-offs, stock dividends, rights offerings, mergers,
special dividends, ticker changes, or delistings, each of which can require
different economic accounting.

# 5. Interview-ready explanation

## Technical version

I studied how incorrect corporate-action treatment can contaminate an equity
research pipeline.

In a synthetic 2-for-1 split, a raw price series created a false $-50\%$
return even though investor wealth was unchanged. That single observation
inflated 20-day volatility by more than thirteen times, generated an
ex-ante z-score near $-46$, created false moving-average signals, and
increased ridge-regression RMSE by more than six times. Removing the two
event-contaminated model rows before fitting made the raw and action-aware
pipelines identical.

I then reproduced the same accounting mechanism with Apple’s 2020 4-for-1
split. Comparing historical prices in different share units produced a
naive return of approximately $-74.15\%$, while share-count accounting and
the split-adjusted close both produced an economic return of $+3.39\%$.

For Coca-Cola dividends, ignoring cash distributions understated three-year
wealth by approximately $9.42\%$. On four of twelve dividend dates, the
price return was negative while total return was non-negative. Alpha Vantage
independently matched all twelve yfinance ex-dividend dates and amounts.

The main conclusion is that adjusted prices are not cosmetic preprocessing.
Corporate-action accounting determines the economic meaning of every
downstream statistic and model input.

## Non-technical version

A stock price can change sharply even when the investor has not lost money.

In a stock split, the price per share falls because the investor receives
more shares. In a dividend, the price may fall because value was paid out as
cash. If those events are treated as ordinary losses, the error can distort
risk estimates, trading indicators, and machine-learning models.

My project shows both effects using synthetic examples and real Apple and
Coca-Cola data. The main lesson is that a price series must be reconciled
with dividends, splits, and share counts before it can be interpreted as an
investor-return series.